# Self Improvement

In [26]:
# Imports and environment first

import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_community.tools import GoogleSerperRun
from langchain_community.utilities import GoogleSerperAPIWrapper
from deepagents import create_deep_agent
from deepagents.backends import FilesystemBackend
from langchain_mcp_adapters.client import MultiServerMCPClient
from langchain.agents import create_agent
from pydantic import BaseModel, Field
from pathlib import Path


load_dotenv(override=True)

True

In [27]:
QUESTION = """
I have selected a secret number between 1 and 1000000.

Your job is to discover it.

After every attempt, you will receive feedback telling you
whether your guess was too high or too low.

Read your previous attempts from the Markdown file.
Use the feedback to narrow down the possible numbers.

Use binary search to solve this efficiently.
Return your guess in the answer field.
"""

MEMORY_FILE = Path("self_improvement.md")
SECRET_NUMBER = 52513
MAX_ATTEMPTS=20

# 1. Structured output from the agent
class Attempt(BaseModel):
    answer: str = Field(description="Candidate number, digits only")
    approach: str = Field(
        description="Brief summary of the approach used"
    )
    improvement: str = Field(
        description="What was improved using previous feedback"
    )

In [28]:
model = ChatOpenAI(
    model="openai/gpt-6.2-astra",
    base_url="https://openrouter.ai/api/v1",
    api_key=os.getenv("OPENROUTER_API_KEY")
)

# 2. Create the agent
agent = create_agent(
    name="Self Improving Agent",
    model=model,
    system_prompt="""
    You are solving a difficult mathematical problem.

    Read the supplied Markdown history before each attempt.

    Learn from previous failures:
    - Identify which constraints failed.
    - Avoid repeating failed candidates.
    - Change your strategy based on the verifier feedback.

    Produce one candidate answer per attempt.

    Provide a concise summary of your approach and
    what you improved. Do not provide private reasoning.
    """,
    response_format=Attempt,
)


# 3. Independent answer verifier
def verify(answer: str) -> tuple[bool, str]:
    try:
        guess = int(answer)
    except ValueError:
        return False, "Your answer must be an integer."

    if not 1 <= guess <= 1000000:
        return False, "Your guess must be between 1 and 1000000."

    if guess < SECRET_NUMBER:
        return False, "Too low. The secret number is higher."

    if guess > SECRET_NUMBER:
        return False, "Too high. The secret number is lower."

    return True, "Correct! You discovered the secret number."


MEMORY_FILE.write_text(
"# Agent Self-Improvement Log\n\n"
f"## Question\n{QUESTION}\n",
encoding="utf-8",
)

for i in range(1, MAX_ATTEMPTS + 1):

    print(f"\n--- Attempt {i} ---")

    # Read the agent's previous attempts
    memory = MEMORY_FILE.read_text(encoding="utf-8")

    # Ask the agent to improve using its memory
    result = await agent.ainvoke({
        "messages": [
            {
                "role": "user",
                "content": f"""
                    Solve this question:

                    {QUESTION}

                    Here is your previous attempt history:

                    {memory}

                    Use the feedback to improve your answer.
                """
            }
        ]
    })

    attempt = result["structured_response"]

    # Check whether the answer is actually correct
    correct, feedback = verify(attempt.answer)

    # Save this attempt and feedback to Markdown
    with MEMORY_FILE.open("a", encoding="utf-8") as file:
        file.write(
            f"\n## Attempt {i}\n\n"
            f"**Answer:** {attempt.answer}\n\n"
            f"**Approach:** {attempt.approach}\n\n"
            f"**Improvement:** {attempt.improvement}\n\n"
            f"**Feedback:** {feedback}\n\n"
            f"**Passed:** {correct}\n"
        )

    print("Answer:", attempt.answer)
    print("Feedback:", feedback)

    if correct:
        print("\nSUCCESS!")
        print("Final answer:", attempt.answer)
        print("Attempts:", i)
        break

else:
    print("\nNo correct answer found within the attempt limit.")

print("Memory saved to:", MEMORY_FILE)



--- Attempt 1 ---
Answer: 500000
Feedback: Too high. The secret number is lower.

--- Attempt 2 ---
Answer: 250000
Feedback: Too high. The secret number is lower.

--- Attempt 3 ---
Answer: 125000
Feedback: Too high. The secret number is lower.

--- Attempt 4 ---
Answer: 62500
Feedback: Too high. The secret number is lower.

--- Attempt 5 ---
Answer: 31250
Feedback: Too low. The secret number is higher.

--- Attempt 6 ---
Answer: 46875
Feedback: Too low. The secret number is higher.

--- Attempt 7 ---
Answer: 54687
Feedback: Too high. The secret number is lower.

--- Attempt 8 ---
Answer: 50781
Feedback: Too low. The secret number is higher.

--- Attempt 9 ---
Answer: 52734
Feedback: Too high. The secret number is lower.

--- Attempt 10 ---
Answer: 51757
Feedback: Too low. The secret number is higher.

--- Attempt 11 ---
Answer: 52245
Feedback: Too low. The secret number is higher.

--- Attempt 12 ---
Answer: 52489
Feedback: Too low. The secret number is higher.

--- Attempt 13 ---
An